In [1]:
# ==========================================
# SECTION 1 - IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import re
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==========================================
# SECTION 2 - LOAD DATASETS
# ==========================================

# QS 2026
qs = pd.read_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\2026 QS World University Rankings.csv"
)

# THE Rankings
the = pd.read_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\THE World University Rankings 2016-2026.csv"
)

# SCImago
scimago = pd.read_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\ScimagoIR 2026 - Overall Rank - Universities.csv",
    sep=";"
)

# Leiden
leiden = pd.read_excel(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\CWTS\CWTS Leiden Ranking Traditional Edition 2025.xlsx",
    sheet_name="Results"
)

# Historical Rankings
times = pd.read_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\WorldRankingHistorical\timesData.csv"
)

cwur = pd.read_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\WorldRankingHistorical\cwurData.csv"
)

shanghai = pd.read_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\WorldRankingHistorical\shanghaiData.csv"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
# ==========================================
# SECTION 3 - VERIFY DATASETS
# ==========================================

datasets = {
    "QS": qs,
    "THE": the,
    "SCImago": scimago,
    "Leiden": leiden,
    "Times": times,
    "CWUR": cwur,
    "Shanghai": shanghai
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

QS: (1501, 30)
THE: (16713, 14)
SCImago: (5491, 6)
Leiden: (286920, 85)
Times: (2603, 14)
CWUR: (2200, 14)
Shanghai: (4897, 11)


In [4]:
# ==========================================
# SECTION 4 - CREATE MASTER DATASET
# ==========================================

master = qs.copy()

master.columns = master.columns.str.strip()

print(master.shape)
master.head()

(1501, 30)


,2026 Rank,Previous Rank,Institution Name,Country/Territory,Region,Size,Focus,Research,Status,AR SCORE,...,ISR RANK,ISD SCORE,ISD RANK,IRN SCORE,IRN RANK,EO SCORE,EO RANK,SUS SCORE,SUS RANK,Overall SCORE
0,1,1,Massachusetts Institute of Technology (MIT),United States of America,Americas,M,CO,VH,Private not for Profit,100.0,...,153,92.3,130,94.1,98,100.0,7,93.8,33,100
1,2,2,Imperial College London,United Kingdom,Europe,L,FO,VH,Public,99.6,...,35,100.0,22,97.5,28,95.9,68,98.3,7=,99.4
2,3,6,Stanford University,United States of America,Americas,L,FC,VH,Private not for Profit,100.0,...,261,76.1,230,96.5,49,100.0,2,95.4,19=,98.9
3,4,3,University of Oxford,United Kingdom,Europe,L,FC,VH,Public,100.0,...,80,98.7,67,100.0,1,100.0,3,77.9,198=,97.9
4,5,4,Harvard University,United States of America,Americas,L,FC,VH,Private not for Profit,100.0,...,217,60.6,335,99.4,8,100.0,1,77.8,201=,97.7


In [5]:
# ==========================================
# STEP 5.1 - RENAME COLUMNS
# ==========================================

# QS
qs = qs.rename(columns={
    "2026 Rank": "qs_rank",
    "Previous Rank": "previous_rank",
    "Overall SCORE": "qs_overall_score",
    "AR SCORE": "academic_reputation",
    "ER SCORE": "employer_reputation",
    "FSR SCORE": "faculty_student_ratio",
    "CPF SCORE": "citations_per_faculty",
    "IFR SCORE": "international_faculty",
    "ISR SCORE": "international_students_score",
    "IRN SCORE": "international_research_network",
    "EO SCORE": "employment_outcomes",
    "SUS SCORE": "sustainability_score"
})

# THE
the = the.rename(columns={
    "Rank": "the_rank",
    "Teaching": "teaching_score",
    "Research Environment": "research_environment",
    "Research Quality": "research_quality",
    "Industry Impact": "industry_impact",
    "International Outlook": "international_outlook"
})

# SCImago
scimago = scimago.rename(columns={
    "Institution": "university",
    "Country": "country",
    "Global Rank": "scimago_global_rank"
})

# Leiden
leiden = leiden.rename(columns={
    "University": "university",
    "Country": "country"
})

print("Columns renamed successfully.")

Columns renamed successfully.


In [6]:
# ==========================================
# STEP 5.2 - STANDARDIZE UNIVERSITY NAMES
# ==========================================

def clean_university(name):
    if pd.isna(name):
        return name

    name = str(name).upper().strip()

    # Remove asterisk
    name = name.replace("*", "")

    # Remove punctuation
    name = re.sub(r"[^\w\s]", "", name)

    # Remove extra spaces
    name = " ".join(name.split())

    return name

datasets = [qs, the, scimago, leiden, times, cwur, shanghai]

for df in datasets:
    if "university" in df.columns:
        df["university"] = df["university"].apply(clean_university)

print("University names standardized.")

University names standardized.


In [7]:
# ==========================================
# STEP 5.3 - STANDARDIZE COUNTRY NAMES
# ==========================================

for df in datasets:
    if "country" in df.columns:
        df["country"] = (
            df["country"]
            .astype(str)
            .str.upper()
            .str.strip()
        )

print("Country names standardized.")

Country names standardized.


In [8]:
# ==========================================
# STEP 5.4 - FILTER THE
# ==========================================

the_latest = the[the["Year"] == 2026].copy()

print(the_latest.shape)

(2191, 14)


In [9]:
# ==========================================
# STEP 5.5 - FILTER LEIDEN
# ==========================================

leiden_latest = leiden[
    (leiden["Period"] == "2020–2023") &
    (leiden["Field"] == "All sciences") &
    (leiden["Frac_counting"] == 0)
].copy()

print(leiden_latest.shape)

(1594, 85)


In [10]:
print("QS:", qs.shape)
print("THE Latest:", the_latest.shape)
print("Leiden Latest:", leiden_latest.shape)
print("SCImago:", scimago.shape)

QS: (1501, 30)
THE Latest: (2191, 14)
Leiden Latest: (1594, 85)
SCImago: (5491, 6)


In [11]:
# ==========================================
# SECTION 6 - STANDARDIZE UNIVERSITY NAMES
# ==========================================

import re

def standardize_university(name):
    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    # Remove special characters
    name = re.sub(r'[*(),.]', '', name)

    # Expand common abbreviations
    replacements = {
        "UNIV ": "UNIVERSITY ",
        "INST ": "INSTITUTE ",
        "TECH ": "TECHNOLOGY ",
        "COLL ": "COLLEGE ",
        "&": "AND"
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    # Remove extra spaces
    name = " ".join(name.split())

    return name

In [12]:
datasets = [qs, the_latest, leiden_latest, scimago, times, cwur, shanghai]

for df in datasets:
    if "university" in df.columns:
        df["merge_key"] = df["university"].apply(standardize_university)

In [13]:
manual_mapping = {
    "MIT": "MASSACHUSETTS INSTITUTE OF TECHNOLOGY",
    "CALTECH": "CALIFORNIA INSTITUTE OF TECHNOLOGY",
    "NUS SINGAPORE": "NATIONAL UNIVERSITY OF SINGAPORE",
    "NTU SINGAPORE": "NANYANG TECHNOLOGICAL UNIVERSITY",
    "UNIV OF OXFORD": "UNIVERSITY OF OXFORD",
    "UNIV OF CAMBRIDGE": "UNIVERSITY OF CAMBRIDGE",
    "UNIV OF CHICAGO": "UNIVERSITY OF CHICAGO",
    "UCL": "UNIVERSITY COLLEGE LONDON",
    "EPFL": "ECOLE POLYTECHNIQUE FEDERALE DE LAUSANNE"
}

for df in datasets:
    if "merge_key" in df.columns:
        df["merge_key"] = df["merge_key"].replace(manual_mapping)

In [15]:
print("QS")
print(qs.columns.tolist())

print("\nTHE")
print(the_latest.columns.tolist())

print("\nLeiden")
print(leiden_latest.columns.tolist())

print("\nSCImago")
print(scimago.columns.tolist())

QS
['qs_rank', 'previous_rank', 'Institution Name', 'Country/Territory', 'Region', 'Size', 'Focus', 'Research', 'Status', 'academic_reputation', 'AR RANK', 'employer_reputation', 'ER RANK', 'faculty_student_ratio', 'FSR RANK', 'citations_per_faculty', 'CPF RANK', 'international_faculty', 'IFR RANK', 'international_students_score', 'ISR RANK', 'ISD SCORE', 'ISD RANK', 'international_research_network', 'IRN RANK', 'employment_outcomes', 'EO RANK', 'sustainability_score', 'SUS RANK', 'qs_overall_score']

THE
['the_rank', 'Name', 'Country', 'Student Population', 'Students to Staff Ratio', 'International Students', 'Female to Male Ratio', 'Overall Score', 'teaching_score', 'research_environment', 'research_quality', 'industry_impact', 'international_outlook', 'Year']

Leiden
['university', 'ROR_ID', 'country', 'Field', 'Period', 'Frac_counting', 'P', 'TCS', 'TNCS', 'P_top_1', 'P_top_5', 'P_top_10', 'P_top_50', 'P_collab', 'P_int_collab', 'P_industry_collab', 'P_short_dist_collab', 'P_long_d

In [16]:
qs = qs.rename(columns={
    "Institution Name": "university",
    "Country/Territory": "country"
})

In [17]:
qs = qs.rename(columns={
    "Institution Name": "university",
    "Country/Territory": "country"
})

In [18]:
the_latest = the_latest.rename(columns={
    "Name": "university",
    "Country": "country",
    "Overall Score": "the_overall_score"
})

In [19]:
the_latest["merge_key"] = the_latest["university"].apply(standardize_university)

In [20]:
print(qs.columns)
print(the_latest.columns)

Index(['qs_rank', 'previous_rank', 'university', 'country', 'Region', 'Size',
       'Focus', 'Research', 'Status', 'academic_reputation', 'AR RANK',
       'employer_reputation', 'ER RANK', 'faculty_student_ratio', 'FSR RANK',
       'citations_per_faculty', 'CPF RANK', 'international_faculty',
       'IFR RANK', 'international_students_score', 'ISR RANK', 'ISD SCORE',
       'ISD RANK', 'international_research_network', 'IRN RANK',
       'employment_outcomes', 'EO RANK', 'sustainability_score', 'SUS RANK',
       'qs_overall_score'],
      dtype='object')
Index(['the_rank', 'university', 'country', 'Student Population',
       'Students to Staff Ratio', 'International Students',
       'Female to Male Ratio', 'the_overall_score', 'teaching_score',
       'research_environment', 'research_quality', 'industry_impact',
       'international_outlook', 'Year', 'merge_key'],
      dtype='object')


In [21]:
print("QS vs THE:",
      len(set(qs["merge_key"]) &
          set(the_latest["merge_key"])))

print("QS vs Leiden:",
      len(set(qs["merge_key"]) &
          set(leiden_latest["merge_key"])))

print("QS vs SCImago:",
      len(set(qs["merge_key"]) &
          set(scimago["merge_key"])))

KeyError: 'merge_key'

In [22]:
print(qs.columns.tolist())

['qs_rank', 'previous_rank', 'university', 'country', 'Region', 'Size', 'Focus', 'Research', 'Status', 'academic_reputation', 'AR RANK', 'employer_reputation', 'ER RANK', 'faculty_student_ratio', 'FSR RANK', 'citations_per_faculty', 'CPF RANK', 'international_faculty', 'IFR RANK', 'international_students_score', 'ISR RANK', 'ISD SCORE', 'ISD RANK', 'international_research_network', 'IRN RANK', 'employment_outcomes', 'EO RANK', 'sustainability_score', 'SUS RANK', 'qs_overall_score']


In [23]:
print(qs[["university"]].head())

                                    university
0  Massachusetts Institute of Technology (MIT)
1                      Imperial College London
2                          Stanford University
3                         University of Oxford
4                           Harvard University


In [24]:
qs["merge_key"] = qs["university"].astype(str).apply(standardize_university)

In [25]:
print(qs.columns.tolist())

['qs_rank', 'previous_rank', 'university', 'country', 'Region', 'Size', 'Focus', 'Research', 'Status', 'academic_reputation', 'AR RANK', 'employer_reputation', 'ER RANK', 'faculty_student_ratio', 'FSR RANK', 'citations_per_faculty', 'CPF RANK', 'international_faculty', 'IFR RANK', 'international_students_score', 'ISR RANK', 'ISD SCORE', 'ISD RANK', 'international_research_network', 'IRN RANK', 'employment_outcomes', 'EO RANK', 'sustainability_score', 'SUS RANK', 'qs_overall_score', 'merge_key']


In [26]:
the_latest["merge_key"] = the_latest["university"].astype(str).apply(standardize_university)

In [27]:
print(the_latest.columns.tolist())

['the_rank', 'university', 'country', 'Student Population', 'Students to Staff Ratio', 'International Students', 'Female to Male Ratio', 'the_overall_score', 'teaching_score', 'research_environment', 'research_quality', 'industry_impact', 'international_outlook', 'Year', 'merge_key']


In [28]:
print(cwur.columns.tolist())

['world_rank', 'institution', 'country', 'national_rank', 'quality_of_education', 'alumni_employment', 'quality_of_faculty', 'publications', 'influence', 'citations', 'broad_impact', 'patents', 'score', 'year']


In [29]:
print(shanghai.columns.tolist())

['world_rank', 'university_name', 'national_rank', 'total_score', 'alumni', 'award', 'hici', 'ns', 'pub', 'pcp', 'year']


In [30]:
print(times.columns.tolist())

['world_rank', 'university_name', 'country', 'teaching', 'international', 'research', 'citations', 'income', 'total_score', 'num_students', 'student_staff_ratio', 'international_students', 'female_male_ratio', 'year']


In [31]:
# CWUR
cwur = cwur.rename(columns={
    "institution": "university"
})

# Shanghai
shanghai = shanghai.rename(columns={
    "university_name": "university"
})

# Times Historical
times = times.rename(columns={
    "university_name": "university"
})

In [33]:
import re

def clean_name(name):
    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    # Remove *
    name = name.replace("*", "")

    # Remove punctuation
    name = re.sub(r"[^\w\s]", "", name)

    # Expand common abbreviations
    name = name.replace("UNIV ", "UNIVERSITY ")
    name = name.replace("INST ", "INSTITUTE ")
    name = name.replace("&", "AND")

    # Remove extra spaces
    name = " ".join(name.split())

    return name

In [34]:
print(clean_name("MIT"))
print(clean_name("University of Oxford *"))

MIT
UNIVERSITY OF OXFORD


In [35]:
qs["merge_key"] = qs["university"].apply(clean_name)

the_latest["merge_key"] = the_latest["university"].apply(clean_name)

leiden_latest["merge_key"] = leiden_latest["university"].apply(clean_name)

scimago["merge_key"] = scimago["university"].apply(clean_name)

cwur["merge_key"] = cwur["university"].apply(clean_name)

shanghai["merge_key"] = shanghai["university"].apply(clean_name)

times["merge_key"] = times["university"].apply(clean_name)

In [36]:
qs["merge_key"] = qs["university"].apply(clean_name)

the_latest["merge_key"] = the_latest["university"].apply(clean_name)

leiden_latest["merge_key"] = leiden_latest["university"].apply(clean_name)

scimago["merge_key"] = scimago["university"].apply(clean_name)

cwur["merge_key"] = cwur["university"].apply(clean_name)

shanghai["merge_key"] = shanghai["university"].apply(clean_name)

times["merge_key"] = times["university"].apply(clean_name)

In [38]:
try:
    final_dataset = pd.merge(
        qs,
        the_latest[
            [
                "merge_key",
                "the_rank",
                "teaching_score",
                "research_environment",
                "research_quality",
                "industry_impact",
                "international_outlook",
                "Student Population",
                "Students to Staff Ratio",
                "International Students",
                "Female to Male Ratio",
                "the_overall_score"
            ]
        ],
        on="merge_key",
        how="left"
    )

    print("Merge Successful!")
    print(final_dataset.shape)

except Exception as e:
    print(e)

Merge Successful!
(1501, 42)


In [39]:
print(the_latest.columns.tolist())

['the_rank', 'university', 'country', 'Student Population', 'Students to Staff Ratio', 'International Students', 'Female to Male Ratio', 'the_overall_score', 'teaching_score', 'research_environment', 'research_quality', 'industry_impact', 'international_outlook', 'Year', 'merge_key']


In [40]:
final_dataset = pd.merge(
    qs,
    the_latest[
        [
            "merge_key",
            "the_rank",
            "teaching_score",
            "research_environment",
            "research_quality",
            "industry_impact",
            "international_outlook",
            "Student Population",
            "Students to Staff Ratio",
            "International Students",
            "Female to Male Ratio",
            "the_overall_score"
        ]
    ],
    on="merge_key",
    how="left"
)

print(final_dataset.shape)

(1501, 42)


In [41]:
print(qs.columns.tolist())

['qs_rank', 'previous_rank', 'university', 'country', 'Region', 'Size', 'Focus', 'Research', 'Status', 'academic_reputation', 'AR RANK', 'employer_reputation', 'ER RANK', 'faculty_student_ratio', 'FSR RANK', 'citations_per_faculty', 'CPF RANK', 'international_faculty', 'IFR RANK', 'international_students_score', 'ISR RANK', 'ISD SCORE', 'ISD RANK', 'international_research_network', 'IRN RANK', 'employment_outcomes', 'EO RANK', 'sustainability_score', 'SUS RANK', 'qs_overall_score', 'merge_key']


In [42]:
print(qs.shape)

(1501, 31)


In [43]:
print("QS merge_key exists:", "merge_key" in qs.columns)
print("THE merge_key exists:", "merge_key" in the_latest.columns)

columns_needed = [
    "merge_key",
    "the_rank",
    "teaching_score",
    "research_environment",
    "research_quality",
    "industry_impact",
    "international_outlook",
    "Student Population",
    "Students to Staff Ratio",
    "International Students",
    "Female to Male Ratio",
    "the_overall_score"
]

print("\nMissing columns:")
for col in columns_needed:
    if col not in the_latest.columns:
        print(col)

QS merge_key exists: True
THE merge_key exists: True

Missing columns:


In [44]:
try:
    final_dataset = pd.merge(
        qs,
        the_latest[columns_needed],
        on="merge_key",
        how="left"
    )

    print("SUCCESS")
    print(final_dataset.shape)

except Exception as e:
    print(type(e).__name__)
    print(e)

SUCCESS
(1501, 42)


In [45]:
final_dataset = pd.merge(
    qs,
    the_latest[[
        "merge_key",
        "the_rank",
        "teaching_score",
        "research_environment",
        "research_quality",
        "industry_impact",
        "international_outlook",
        "Student Population",
        "Students to Staff Ratio",
        "International Students",
        "Female to Male Ratio",
        "the_overall_score"
    ]],
    on="merge_key",
    how="left"
)

In [46]:
print(final_dataset.shape)

(1501, 42)


In [47]:
print(final_dataset.shape)

(1501, 42)


In [48]:
final_dataset = final_dataset.merge(
    leiden_latest[
        [
            "merge_key",
            "P",
            "TCS",
            "MNCS",
            "P_top_10",
            "P_collab",
            "P_int_collab",
            "P_industry_collab",
            "PP_OA",
            "PA_M",
            "PA_F"
        ]
    ],
    on="merge_key",
    how="left"
)

In [49]:
final_dataset.shape

(1502, 52)

In [50]:
final_dataset = final_dataset.merge(
    scimago[
        [
            "merge_key",
            "scimago_global_rank",
            "Best Country Quartile"
        ]
    ],
    on="merge_key",
    how="left"
)

In [51]:
final_dataset.shape

(1502, 54)

In [52]:
final_dataset = final_dataset.merge(
    cwur,
    on="merge_key",
    how="left",
    suffixes=("", "_cwur")
)

In [53]:
final_dataset.shape

(2095, 68)

In [54]:
final_dataset = final_dataset.merge(
    shanghai,
    on="merge_key",
    how="left",
    suffixes=("", "_shanghai")
)

In [55]:
final_dataset.shape

(7356, 79)

In [56]:
final_dataset = final_dataset.merge(
    times,
    on="merge_key",
    how="left",
    suffixes=("", "_times")
)

In [57]:
final_dataset.shape

(28864, 93)

In [58]:
final_dataset.to_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\merged_raw_dataset.csv",
    index=False
)

In [59]:
print(final_dataset.shape)

(28864, 93)


In [60]:
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28864 entries, 0 to 28863
Data columns (total 93 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   qs_rank                         28864 non-null  object 
 1   previous_rank                   28708 non-null  object 
 2   university                      28864 non-null  object 
 3   country                         28864 non-null  object 
 4   Region                          28864 non-null  object 
 5   Size                            28863 non-null  object 
 6   Focus                           28864 non-null  object 
 7   Research                        28863 non-null  object 
 8   Status                          28676 non-null  object 
 9   academic_reputation             28864 non-null  float64
 10  AR RANK                         28864 non-null  object 
 11  employer_reputation             28864 non-null  float64
 12  ER RANK                         

In [61]:
print(final_dataset.columns.tolist())

['qs_rank', 'previous_rank', 'university', 'country', 'Region', 'Size', 'Focus', 'Research', 'Status', 'academic_reputation', 'AR RANK', 'employer_reputation', 'ER RANK', 'faculty_student_ratio', 'FSR RANK', 'citations_per_faculty', 'CPF RANK', 'international_faculty', 'IFR RANK', 'international_students_score', 'ISR RANK', 'ISD SCORE', 'ISD RANK', 'international_research_network', 'IRN RANK', 'employment_outcomes', 'EO RANK', 'sustainability_score', 'SUS RANK', 'qs_overall_score', 'merge_key', 'the_rank', 'teaching_score', 'research_environment', 'research_quality', 'industry_impact', 'international_outlook', 'Student Population', 'Students to Staff Ratio', 'International Students', 'Female to Male Ratio', 'the_overall_score', 'P', 'TCS', 'MNCS', 'P_top_10', 'P_collab', 'P_int_collab', 'P_industry_collab', 'PP_OA', 'PA_M', 'PA_F', 'scimago_global_rank', 'Best Country Quartile', 'world_rank', 'university_cwur', 'country_cwur', 'national_rank', 'quality_of_education', 'alumni_employment

In [62]:
missing = final_dataset.isnull().sum().sort_values(ascending=False)
print(missing.head(30))

total_score                17288
broad_impact                7643
Female to Male Ratio        5567
female_male_ratio           5272
Best Country Quartile       4262
scimago_global_rank         4262
industry_impact             3105
teaching_score              3105
Student Population          3105
research_environment        3105
Students to Staff Ratio     3105
International Students      3105
the_overall_score           3105
the_rank                    3105
research_quality            3105
international_outlook       3105
P_industry_collab           2142
P_top_10                    2142
P_int_collab                2142
MNCS                        2142
TCS                         2142
P                           2142
PA_M                        2142
P_collab                    2142
PA_F                        2142
PP_OA                       2142
ns                          1886
hici                        1866
award                       1866
pub                         1866
dtype: int

In [64]:
print(final_dataset[["qs_overall_score", "the_overall_score", "score"]].dtypes)

qs_overall_score      object
the_overall_score    float64
score                float64
dtype: object


In [65]:
final_dataset["qs_overall_score"] = pd.to_numeric(
    final_dataset["qs_overall_score"],
    errors="coerce"
)

final_dataset["the_overall_score"] = pd.to_numeric(
    final_dataset["the_overall_score"],
    errors="coerce"
)

final_dataset["score"] = pd.to_numeric(
    final_dataset["score"],
    errors="coerce"
)

In [66]:
print(final_dataset[["qs_overall_score", "the_overall_score", "score"]].dtypes)

qs_overall_score     float64
the_overall_score    float64
score                float64
dtype: object


In [67]:
final_dataset["Overall Performance Index"] = (
    final_dataset["qs_overall_score"].fillna(0) * 0.4 +
    final_dataset["the_overall_score"].fillna(0) * 0.3 +
    final_dataset["score"].fillna(0) * 0.3
)

In [68]:
print(final_dataset.shape)

(28864, 95)


In [69]:
print(final_dataset.shape)

(28864, 95)


In [70]:
print("QS:", qs.shape)
print("After THE:", the_latest.shape)
print("Leiden:", leiden_latest.shape)
print("SCImago:", scimago.shape)
print("CWUR:", cwur.shape)
print("Shanghai:", shanghai.shape)
print("Times:", times.shape)

QS: (1501, 31)
After THE: (2191, 15)
Leiden: (1594, 86)
SCImago: (5491, 7)
CWUR: (2200, 15)
Shanghai: (4897, 12)
Times: (2603, 15)


In [71]:
print(final_dataset["merge_key"].value_counts().head(20))

merge_key
UNIVERSITY OF CHICAGO                        264
CARNEGIE MELLON UNIVERSITY                   264
NORTHWESTERN UNIVERSITY                      264
BROWN UNIVERSITY                             264
UNIVERSITY OF ILLINOIS AT URBANACHAMPAIGN    264
DUKE UNIVERSITY                              264
KYOTO UNIVERSITY                             264
UNIVERSITY OF CALIFORNIA IRVINE              264
UNIVERSITY OF ARIZONA                        264
UNIVERSITY OF BRITISH COLUMBIA               264
PRINCETON UNIVERSITY                         264
MCGILL UNIVERSITY                            264
VANDERBILT UNIVERSITY                        264
UNIVERSITY OF CALIFORNIA DAVIS               264
UNIVERSITY OF HELSINKI                       264
BOSTON UNIVERSITY                            264
UNIVERSITY OF COPENHAGEN                     264
UNIVERSITY OF UTAH                           264
UNIVERSITY OF GENEVA                         264
WASHINGTON UNIVERSITY IN ST LOUIS            264
Name: coun

In [72]:
cwur_latest = cwur[cwur["year"] == cwur["year"].max()].copy()

times_latest = times[times["year"] == times["year"].max()].copy()

shanghai_latest = shanghai[shanghai["year"] == shanghai["year"].max()].copy()

In [73]:
final_dataset = pd.merge(
    qs,
    the_latest[
        [
            "merge_key",
            "the_rank",
            "teaching_score",
            "research_environment",
            "research_quality",
            "industry_impact",
            "international_outlook",
            "Student Population",
            "Students to Staff Ratio",
            "International Students",
            "Female to Male Ratio",
            "the_overall_score"
        ]
    ],
    on="merge_key",
    how="left"
)

In [74]:
print(final_dataset.shape)

(1501, 42)


In [75]:
final_dataset = final_dataset.merge(
    leiden_latest[
        [
            "merge_key",
            "P",
            "TCS",
            "MNCS",
            "P_top_10",
            "P_collab",
            "P_int_collab",
            "P_industry_collab",
            "PP_OA",
            "PA_M",
            "PA_F"
        ]
    ],
    on="merge_key",
    how="left"
)

In [76]:
print(final_dataset.shape)

(1502, 52)


In [77]:
final_dataset = final_dataset.merge(
    scimago[
        [
            "merge_key",
            "scimago_global_rank",
            "Best Country Quartile"
        ]
    ],
    on="merge_key",
    how="left"
)

In [78]:
print(final_dataset.shape)

(1502, 54)


In [79]:
cwur_latest = cwur[cwur["year"] == cwur["year"].max()].copy()

times_latest = times[times["year"] == times["year"].max()].copy()

shanghai_latest = shanghai[shanghai["year"] == shanghai["year"].max()].copy()

In [80]:
print(cwur_latest.shape)
print(times_latest.shape)
print(shanghai_latest.shape)

(1000, 15)
(800, 15)
(500, 12)


In [81]:
cwur_latest = cwur_latest.drop_duplicates(subset="merge_key")

times_latest = times_latest.drop_duplicates(subset="merge_key")

shanghai_latest = shanghai_latest.drop_duplicates(subset="merge_key")

In [82]:
print(cwur_latest.shape)
print(times_latest.shape)
print(shanghai_latest.shape)

(1000, 15)
(800, 15)
(500, 12)


In [83]:
final_dataset = final_dataset.merge(
    cwur_latest[
        [
            "merge_key",
            "national_rank",
            "quality_of_education",
            "alumni_employment",
            "quality_of_faculty",
            "publications",
            "influence",
            "citations",
            "broad_impact",
            "patents",
            "score"
        ]
    ],
    on="merge_key",
    how="left"
)

print(final_dataset.shape)

(1502, 64)


In [84]:
final_dataset = final_dataset.merge(
    shanghai_latest[
        [
            "merge_key",
            "total_score",
            "alumni",
            "award",
            "hici",
            "ns",
            "pub",
            "pcp"
        ]
    ],
    on="merge_key",
    how="left"
)

print(final_dataset.shape)

(1502, 71)


In [85]:
final_dataset = final_dataset.merge(
    times_latest[
        [
            "merge_key",
            "teaching",
            "research",
            "citations",
            "income",
            "total_score"
        ]
    ],
    on="merge_key",
    how="left",
    suffixes=("", "_times")
)

print(final_dataset.shape)

(1502, 76)


In [86]:
final_dataset = final_dataset.merge(
    times_latest[
        [
            "merge_key",
            "teaching",
            "research",
            "citations",
            "income",
            "total_score"
        ]
    ],
    on="merge_key",
    how="left",
    suffixes=("", "_times")
)

print(final_dataset.shape)

(1502, 81)


In [87]:
print(final_dataset["merge_key"].duplicated().sum())

1


In [88]:
final_dataset.to_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset.csv",
    index=False
)

final_dataset.to_excel(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset.xlsx",
    index=False
)

In [89]:
print(final_dataset.shape)

(1502, 81)


In [90]:
print(final_dataset["merge_key"].duplicated().sum())

1


In [91]:
final_dataset = final_dataset.drop_duplicates(subset="merge_key")

In [92]:
print(final_dataset.shape)

(1501, 81)


In [93]:
for col in final_dataset.columns:
    if col.endswith("_x") or col.endswith("_y") or col.endswith("_times"):
        print(col)

citations_times
total_score_times
teaching_times
research_times
citations_times
income_times
total_score_times


In [94]:
missing = final_dataset.isnull().sum().sort_values(ascending=False)

print(missing.head(20))

total_score             1442
award                   1216
ns                      1216
pcp                     1216
hici                    1216
pub                     1216
alumni                  1216
patents                 1022
quality_of_education    1022
citations               1022
broad_impact            1022
national_rank           1022
alumni_employment       1022
publications            1022
influence               1022
quality_of_faculty      1022
score                   1022
citations_times         1013
research                1013
income                  1013
dtype: int64


In [95]:
final_dataset.to_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset.csv",
    index=False
)

final_dataset.to_excel(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset.xlsx",
    index=False
)

In [96]:
final_dataset = final_dataset.loc[:, ~final_dataset.columns.duplicated()]

In [97]:
print(final_dataset.shape)

(1501, 79)


In [98]:
numeric_cols = [
    "qs_overall_score",
    "the_overall_score",
    "academic_reputation",
    "employer_reputation",
    "faculty_student_ratio",
    "citations_per_faculty",
    "international_faculty",
    "international_students_score",
    "international_research_network",
    "employment_outcomes",
    "sustainability_score",
    "teaching_score",
    "research_environment",
    "research_quality",
    "industry_impact",
    "international_outlook",
    "P",
    "TCS",
    "MNCS",
    "score",
    "total_score"
]

for col in numeric_cols:
    if col in final_dataset.columns:
        final_dataset[col] = pd.to_numeric(final_dataset[col], errors="coerce")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27404\2094675453.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_dataset[col] = pd.to_numeric(final_dataset[col], errors="coerce")


In [99]:
final_dataset = final_dataset.copy()

In [100]:
numeric_cols = [
    "qs_overall_score",
    "the_overall_score",
    "academic_reputation",
    "employer_reputation",
    "faculty_student_ratio",
    "citations_per_faculty",
    "international_faculty",
    "international_students_score",
    "international_research_network",
    "employment_outcomes",
    "sustainability_score",
    "teaching_score",
    "research_environment",
    "research_quality",
    "industry_impact",
    "international_outlook",
    "P",
    "TCS",
    "MNCS",
    "score",
    "total_score"
]

for col in numeric_cols:
    if col in final_dataset.columns:
        final_dataset[col] = pd.to_numeric(final_dataset[col], errors="coerce")

In [101]:
final_dataset[numeric_cols].dtypes

qs_overall_score                  float64
the_overall_score                 float64
academic_reputation               float64
employer_reputation               float64
faculty_student_ratio             float64
citations_per_faculty             float64
international_faculty             float64
international_students_score      float64
international_research_network    float64
employment_outcomes               float64
sustainability_score              float64
teaching_score                    float64
research_environment              float64
research_quality                  float64
industry_impact                   float64
international_outlook             float64
P                                 float64
TCS                               float64
MNCS                              float64
score                             float64
total_score                       float64
dtype: object

In [102]:
final_dataset.to_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset.csv",
    index=False
)

final_dataset.to_excel(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset.xlsx",
    index=False
)

print("Export successful!")

Export successful!


In [104]:
print(final_dataset["qs_rank"].dtype)

object


In [105]:
final_dataset["qs_rank"] = (
    final_dataset["qs_rank"]
    .astype(str)
    .str.extract(r'(\d+)')[0]
)

final_dataset["qs_rank"] = pd.to_numeric(
    final_dataset["qs_rank"],
    errors="coerce"
)

In [106]:
print(final_dataset["qs_rank"].dtype)

int64


In [107]:
final_dataset["Country Average Rank"] = (
    final_dataset.groupby("country")["qs_rank"].transform("mean")
)

In [108]:
final_dataset["Country Average Rank"] = (
    final_dataset.groupby("country")["qs_rank"].transform("mean")
)

In [109]:
final_dataset["Country Average Score"] = (
    final_dataset.groupby("country")["qs_overall_score"].transform("mean")
)

In [110]:
final_dataset["qs_overall_score"] = (
    pd.to_numeric(final_dataset["qs_overall_score"], errors="coerce")
)

In [111]:
final_dataset["Universities in Country"] = (
    final_dataset.groupby("country")["university"].transform("count")
)

In [112]:
final_dataset["Best University Rank"] = (
    final_dataset.groupby("country")["qs_rank"].transform("min")
)

In [113]:
final_dataset["Global Ranking Score"] = (
    1502 - final_dataset["qs_rank"]
)

In [114]:
final_dataset["Research Productivity Index"] = (
    final_dataset["P"].fillna(0)
)

In [115]:
final_dataset["Citation Impact Score"] = (
    final_dataset["MNCS"].fillna(0)
)

In [116]:
final_dataset["Internationalization Score"] = (
    final_dataset["international_students_score"].fillna(0) +
    final_dataset["international_faculty"].fillna(0)
) / 2

In [117]:
final_dataset["Research Excellence Score"] = (
    final_dataset["research_quality"].fillna(0) * 0.4 +
    final_dataset["research_environment"].fillna(0) * 0.3 +
    final_dataset["teaching_score"].fillna(0) * 0.3
)

In [118]:
final_dataset["Overall Performance Index"] = (
    final_dataset["qs_overall_score"].fillna(0) * 0.4 +
    final_dataset["the_overall_score"].fillna(0) * 0.3 +
    final_dataset["score"].fillna(0) * 0.3
)

In [119]:
print(final_dataset.shape)

(1501, 89)


In [120]:
final_dataset.to_csv(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset_v2.csv",
    index=False
)

final_dataset.to_excel(
    r"C:\Users\ADMIN\Desktop\Muskan Programs\EduVision\data\university_final_dataset_v2.xlsx",
    index=False
)

print("✅ Final dataset exported successfully!")

✅ Final dataset exported successfully!


In [121]:
print(final_dataset.info())
print(final_dataset.isnull().sum().sort_values(ascending=False).head(20))

<class 'pandas.core.frame.DataFrame'>
Index: 1501 entries, 0 to 1501
Data columns (total 89 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   qs_rank                         1501 non-null   int64  
 1   previous_rank                   1389 non-null   object 
 2   university                      1501 non-null   object 
 3   country                         1501 non-null   object 
 4   Region                          1501 non-null   object 
 5   Size                            1500 non-null   object 
 6   Focus                           1501 non-null   object 
 7   Research                        1500 non-null   object 
 8   Status                          1454 non-null   object 
 9   academic_reputation             1501 non-null   float64
 10  AR RANK                         1501 non-null   object 
 11  employer_reputation             1501 non-null   float64
 12  ER RANK                         1501 no